<a href="https://colab.research.google.com/github/amanshaikh45975-oss/Advanced-Bank-Term-Deposit-Analysis/blob/main/Netflix_Analysis_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Importing Libraries**

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

### **Reading The Dataset**

In [ ]:
df = pd.read_csv('netflix1.csv')

## **Reviewing Dataset**

In [ ]:
#first top 3 values
df.head(10)                            #the dataset is Already In Snake Case soo it will be easier to manipulate using excel w+ithout any error.

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"
5,s9,TV Show,The Great British Baking Show,Andy Devonshire,United Kingdom,9/24/2021,2021,TV-14,9 Seasons,"British TV Shows, Reality TV"
6,s10,Movie,The Starling,Theodore Melfi,United States,9/24/2021,2021,PG-13,104 min,"Comedies, Dramas"
7,s939,Movie,Motu Patlu in the Game of Zones,Suhas Kadav,India,5/1/2021,2019,TV-Y7,87 min,"Children & Family Movies, Comedies, Music & Mu..."
8,s13,Movie,Je Suis Karl,Christian Schwochow,Germany,9/23/2021,2021,TV-MA,127 min,"Dramas, International Movies"
9,s940,Movie,Motu Patlu in Wonderland,Suhas Kadav,India,5/1/2021,2013,TV-Y7,76 min,"Children & Family Movies, Music & Musicals"


In [ ]:
df.describe(include='all')            #Both numerical and categorical basic statistical values of dataType

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
count,8790,8790,8790,8790,8790,8790,8790.000000,8790,8790,8790
unique,8790,2,8787,4528,86,1713,NaN,14,220,513
top,s8786,Movie,9-Feb,Not Given,United States,1/1/2020,NaN,TV-MA,1 Season,"Dramas, International Movies"
freq,1,6126,2,2588,3240,110,NaN,3205,1791,362
mean,NaN,NaN,NaN,NaN,NaN,NaN,2014.183163,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,8.825466,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,1925.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,2013.000000,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,2017.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,2019.000000,NaN,NaN,NaN


In [ ]:
df.shape                   # no of rows = 8790 , columns = 10

(8790, 10)

In [ ]:
df.info()                  #Date is stored In string format or not

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   country       8790 non-null   object
 5   date_added    8790 non-null   object
 6   release_year  8790 non-null   int64 
 7   rating        8790 non-null   object
 8   duration      8790 non-null   object
 9   listed_in     8790 non-null   object
dtypes: int64(1), object(9)
memory usage: 686.8+ KB


In [ ]:
def data_quality_check(df):                                                     #Identify data quality issues
    quality_report = {
        'total_records': len(df),
        'duplicate_rows': df.duplicated().sum(),
        'missing_values': df.isnull().sum().to_dict(),
    }
    return quality_report

quality_report = data_quality_check(df)

In [ ]:
print(quality_report)

{'total_records': 8790, 'duplicate_rows': np.int64(0), 'missing_values': {'show_id': 0, 'type': 0, 'title': 0, 'director': 0, 'country': 0, 'date_added': 0, 'release_year': 0, 'rating': 0, 'duration': 0, 'listed_in': 0}}


**Data Cleaning**

In [ ]:
df_clean = df.copy()                                                            #Remove duplicates
initial_count = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['show_id'], keep='first')
duplicates_removed = initial_count - len(df_clean)

In [ ]:
print(f"Duplicates removed: {duplicates_removed}")

Duplicates removed: 0


In [ ]:
df_clean['type'] = df_clean['type'].str.strip().str.title()                     #Standardize text formatting
df_clean['title'] = df_clean['title'].str.strip().str.title()
df_clean['director'] = df_clean['director'].str.strip().str.title()
df_clean['country'] = df_clean['country'].str.strip().str.title()
df_clean['rating'] = df_clean['rating'].str.strip().str.title()
df_clean['listed_in'] = df_clean['listed_in'].str.strip().str.title()

In [ ]:
print(df_clean[['type','title','director','country','rating','listed_in']].head())

      type                             title         director        country  \
0    Movie              Dick Johnson Is Dead  Kirsten Johnson  United States   
1  Tv Show                         Ganglands  Julien Leclercq         France   
2  Tv Show                     Midnight Mass    Mike Flanagan  United States   
3    Movie  Confessions Of An Invisible Girl    Bruno Garotti         Brazil   
4    Movie                           Sankofa     Haile Gerima  United States   

  rating                                          listed_in  
0  Pg-13                                      Documentaries  
1  Tv-Ma  Crime Tv Shows, International Tv Shows, Tv Act...  
2  Tv-Ma                 Tv Dramas, Tv Horror, Tv Mysteries  
3  Tv-Pg                 Children & Family Movies, Comedies  
4  Tv-Ma   Dramas, Independent Movies, International Movies  


In [ ]:
#Extract just the numbers from the duration column using string extraction.
df_clean['duration_num'] = df_clean['duration'].str.extract('(\d+)').astype(float)

df_clean['movie_duration_mins'] = np.where(df_clean['type'] == 'Movie', df_clean['duration_num'], np.nan) #Creating dedicated Movie duration column (leave TV shows as NaN here).
df_clean['movie_duration_mins'] = df_clean['movie_duration_mins'].fillna(0)

df_clean['tv_show_seasons'] = np.where(df_clean['type'] == 'Tv Show', df_clean['duration_num'], np.nan)  # Creating dedicated TV Show duration column (leave Movies as NaN here).
df_clean['tv_show_seasons'] = df_clean['tv_show_seasons'].fillna(0)

df_clean = df_clean.drop(columns=['duration_num'])                              # Drop the messy temporary extraction column.

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_9603/3154645949.py:2: SyntaxWarning: invalid escape sequence '\d'
  df_clean['duration_num'] = df_clean['duration'].str.extract('(\d+)').astype(float)


In [ ]:
df_clean.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,movie_duration_mins,tv_show_seasons
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,Pg-13,90 min,Documentaries,90.0,0.0
1,s3,Tv Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,Tv-Ma,1 Season,"Crime Tv Shows, International Tv Shows, Tv Act...",0.0,1.0
2,s6,Tv Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,Tv-Ma,1 Season,"Tv Dramas, Tv Horror, Tv Mysteries",0.0,1.0
3,s14,Movie,Confessions Of An Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,Tv-Pg,91 min,"Children & Family Movies, Comedies",91.0,0.0
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,Tv-Ma,125 min,"Dramas, Independent Movies, International Movies",125.0,0.0


### **Creating New Columns**

**Content target column**

In [ ]:
# Analysis by age group
df_clean['rating'] = df_clean['rating'].astype(str).str.strip().str.upper()

kids_rating = ['TV-Y', 'TV-Y7', 'G', 'TV-G', 'TV-Y7-FV']
teen_rating = ['PG', 'TV-PG', 'PG-13', 'TV-14']
audlts_rating = ['TV-MA', 'R', 'NC-17']

df_clean['content_target'] = np.select(
    [
        df_clean['rating'].isin(kids_rating),
        df_clean['rating'].isin(teen_rating),
        df_clean['rating'].isin(audlts_rating)
    ],
    ['kids' , 'Teen' , 'Adults'],
    default = 'Unrated'
)

**Season Count bucket**

In [ ]:
df_clean['season_bucket'] = np.select(
    [
        df_clean['tv_show_seasons'].between(1,2),
        df_clean['tv_show_seasons'].between(3,5),
        df_clean['tv_show_seasons'] >=6
    ],
    ['Short-form (1-2 Seasons)', 'Standard (3-5 Seasons)', 'Long-running (6+ Seasons)'],
    default = 'Movie'
)

In [ ]:
df_clean['tv_show_seasons'].dtype

dtype('float64')

**Release Delay Matric**

In [ ]:
df_clean['year_added'] = pd.to_datetime(df_clean['date_added']).dt.year
df_clean['release_delay_years'] = df_clean['year_added'] - df_clean['release_year']

**Primary Genre**

In [ ]:
df_clean['primary_genre'] = df_clean['listed_in'].str.split(',').str[0].str.strip()

**Quick Validation Check**

In [ ]:
print(df_clean[['title', 'content_target', 'season_bucket', 'release_delay_years', 'primary_genre']].head())

                              title content_target             season_bucket  \
0              Dick Johnson Is Dead           Teen                     Movie   
1                         Ganglands         Adults  Short-form (1-2 Seasons)   
2                     Midnight Mass         Adults  Short-form (1-2 Seasons)   
3  Confessions Of An Invisible Girl           Teen                     Movie   
4                           Sankofa         Adults                     Movie   

   release_delay_years             primary_genre  
0                    1             Documentaries  
1                    0            Crime Tv Shows  
2                    0                 Tv Dramas  
3                    0  Children & Family Movies  
4                   28                    Dramas  


# **Exploratory Data Analysis**

### **Load and Prepare**

In [ ]:
df_clean.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,movie_duration_mins,tv_show_seasons,content_target,season_bucket,year_added,release_delay_years,primary_genre
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries,90.0,0.0,Teen,Movie,2021,1,Documentaries
1,s3,Tv Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime Tv Shows, International Tv Shows, Tv Act...",0.0,1.0,Adults,Short-form (1-2 Seasons),2021,0,Crime Tv Shows
2,s6,Tv Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"Tv Dramas, Tv Horror, Tv Mysteries",0.0,1.0,Adults,Short-form (1-2 Seasons),2021,0,Tv Dramas
3,s14,Movie,Confessions Of An Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies",91.0,0.0,Teen,Movie,2021,0,Children & Family Movies
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",125.0,0.0,Adults,Movie,2021,28,Dramas


In [ ]:
df_clean.to_csv('netflix_cleaned.csv', index=False)
from google.colab import files
files.download('netflix_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>